# Survival Support Vector Machine (R)

Ranking-based survival SVM using `survivalsvm` (Van Belle et al. ranking approach),
with regularisation parameter `gamma.mu` tuned by 5-fold cross-validation.

| Python (scikit-survival) | R (survivalsvm) |
|--------------------------|-----------------|
| `FastSurvivalSVM(rank_ratio=1.0)` | `survivalsvm(type='vanbelle2')` |
| alpha grid 2^−10 to 2^10 | gamma.mu grid 2^−10 to 2^10 |
| `RepeatedKFold(5, 5)` | `KFold(5)` per gamma |
| `permutation_importance` | manual permutation loop |

**Libraries:** `survivalsvm`, `survival`, `survminer`, `ggplot2`, `pheatmap`

In [ ]:
# install.packages(c('survivalsvm','survival','survminer','ggplot2','dplyr','pheatmap','ggpubr'))

In [ ]:
suppressPackageStartupMessages({
  library(survival)
  library(survivalsvm)
  library(survminer)
  library(ggplot2)
  library(dplyr)
  library(pheatmap)
  library(ggpubr)
})

DATA_DIR  <- '../../datasets/csv_files'
VIS_DIR   <- '../../visuals'
MODEL_DIR <- '../../models'
dir.create(VIS_DIR,   showWarnings=FALSE, recursive=TRUE)
dir.create(MODEL_DIR, showWarnings=FALSE, recursive=TRUE)
set.seed(42)
message('Packages loaded.')

## Preparing Data

In [ ]:
load_surv <- function(path) {
  df <- read.csv(path, stringsAsFactors=FALSE)
  df[complete.cases(df[c('relapse_free_time','relapse_free_event')]), ]
}

train_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/train_data.csv'))
test1_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_one.csv'))
test2_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_two.csv'))
test3_raw <- load_surv(file.path(DATA_DIR, 'ml_datasets/test_data_three.csv'))

surv_meta <- c('sample_name','relapse_free_time','relapse_free_event')
gene_cols <- setdiff(names(train_raw), surv_meta)
train_raw$relapse_free_event <- as.integer(train_raw$relapse_free_event)

prep_df <- function(df, genes) {
  avail <- intersect(genes, names(df))
  out   <- df[, c('relapse_free_time','relapse_free_event', avail)]
  out$relapse_free_event <- as.integer(out$relapse_free_event)
  out
}

train_df <- prep_df(train_raw, gene_cols)
test1_df <- prep_df(test1_raw, gene_cols)
test2_df <- prep_df(test2_raw, gene_cols)
test3_df <- prep_df(test3_raw, gene_cols)

cat(sprintf('Train: %d | Test1: %d | Test2: %d | Test3: %d | Genes: %d\n',
            nrow(train_df), nrow(test1_df), nrow(test2_df),
            nrow(test3_df), length(gene_cols)))

surv_formula <- Surv(relapse_free_time, relapse_free_event) ~ .

## Initial Model (default gamma.mu)

In [ ]:
get_ci <- function(model, data) {
  pred <- predict(model, newdata=data)
  concordance(Surv(data$relapse_free_time, data$relapse_free_event) ~
                pred$predicted)$concordance
}

cat('Fitting initial survivalsvm (type=vanbelle2, gamma.mu=0.1)...\n')
svm_init <- survivalsvm(formula=surv_formula, data=train_df,
                        type='vanbelle2', gamma.mu=0.1, opt.meth='ipop')

ci_init <- get_ci(svm_init, train_df)
cat(sprintf('Initial train C-index: %.4f\n', ci_init))

## Tune gamma.mu via 5-Fold Cross-Validation

Grid: 2^(−10) to 2^10 in steps of 2 — identical range to Python alpha grid.

In [ ]:
gamma_grid  <- 2^seq(-10, 10, by=2)
n           <- nrow(train_df)
K           <- 5
folds       <- sample(rep(seq_len(K), length.out=n))
cv_cindices <- numeric(length(gamma_grid))

cat(sprintf('Tuning gamma.mu over %d values (5-fold CV)...\n', length(gamma_grid)))

for (i in seq_along(gamma_grid)) {
  fold_ci <- numeric(K)
  for (k in seq_len(K)) {
    tryCatch({
      idx_val   <- which(folds == k)
      idx_train <- which(folds != k)
      m  <- survivalsvm(formula=surv_formula, data=train_df[idx_train, ],
                        type='vanbelle2', gamma.mu=gamma_grid[i], opt.meth='ipop')
      pv <- predict(m, newdata=train_df[idx_val, ])
      fold_ci[k] <- concordance(
        Surv(train_df$relapse_free_time[idx_val],
             train_df$relapse_free_event[idx_val]) ~ pv$predicted)$concordance
    }, error=function(e) { fold_ci[k] <<- 0.5 })
  }
  cv_cindices[i] <- mean(fold_ci, na.rm=TRUE)
  cat(sprintf('  gamma.mu=%.6f  CV C-index=%.4f\n', gamma_grid[i], cv_cindices[i]))
}

best_gamma <- gamma_grid[which.max(cv_cindices)]
best_cv_ci <- max(cv_cindices)
cat(sprintf('\nBest gamma.mu: %.6f  (CV C-index: %.4f)\n', best_gamma, best_cv_ci))

In [ ]:
cv_df <- data.frame(log2_gamma=log2(gamma_grid), cv_ci=cv_cindices)

ggplot(cv_df, aes(x=log2_gamma, y=cv_ci)) +
  geom_line(colour='#4DBBD5') +
  geom_point(colour='#4DBBD5', size=2) +
  geom_point(data=cv_df[which.max(cv_cindices), ],
             colour='#E64B35', size=4) +
  geom_vline(xintercept=log2(best_gamma), colour='#E64B35', linetype='dashed') +
  geom_hline(yintercept=0.5, colour='grey50', linetype='dashed') +
  labs(title='Survival SVM — CV C-index vs gamma.mu',
       x=expression(log[2](gamma.mu)), y='C-index (5-fold CV)') +
  theme_bw(base_size=10) +
  theme(plot.title=element_text(face='bold'))

## Final Model with Best gamma.mu

In [ ]:
cat(sprintf('Fitting final survivalsvm with gamma.mu=%.6f...\n', best_gamma))
svm_final <- survivalsvm(formula=surv_formula, data=train_df,
                         type='vanbelle2', gamma.mu=best_gamma, opt.meth='ipop')

ci_train <- get_ci(svm_final, train_df)
ci_t1    <- get_ci(svm_final, test1_df)
ci_t2    <- get_ci(svm_final, test2_df)
ci_t3    <- get_ci(svm_final, test3_df)

cat(sprintf('Final SVM C-indices:\n'))
cat(sprintf('  Train:  %.4f\n', ci_train))
cat(sprintf('  Test 1: %.4f\n', ci_t1))
cat(sprintf('  Test 2: %.4f\n', ci_t2))
cat(sprintf('  Test 3: %.4f\n', ci_t3))

## Permutation Importance

For each gene: shuffle its values, measure the drop in C-index — the larger the
drop, the more important the gene.

In [ ]:
perm_importance <- function(model, data, genes, n_perm=3) {
  baseline <- get_ci(model, data)
  imps <- numeric(length(genes))
  for (i in seq_along(genes)) {
    drops <- numeric(n_perm)
    for (p in seq_len(n_perm)) {
      d_perm        <- data
      d_perm[[genes[i]]] <- sample(d_perm[[genes[i]]])
      drops[p] <- baseline - tryCatch(get_ci(model, d_perm), error=function(e) baseline)
    }
    imps[i] <- mean(drops)
  }
  data.frame(gene=genes, importance=imps, stringsAsFactors=FALSE)
}

# Run on top-variance genes for speed
gene_vars    <- apply(train_df[, gene_cols], 2, var)
top_genes_v  <- names(sort(gene_vars, decreasing=TRUE))[seq_len(min(50, length(gene_cols)))]
cat(sprintf('Running permutation on %d top-variance genes...\n', length(top_genes_v)))

vimp_df <- perm_importance(svm_final, train_df, top_genes_v, n_perm=3)
vimp_df <- vimp_df[order(vimp_df$importance, decreasing=TRUE), ]

cat('Top 15 genes by permutation importance:\n')
print(head(vimp_df, 15))

sig_genes <- head(vimp_df$gene, 8)
cat(sprintf('\nSVM Gene Signature (8 genes): %s\n', paste(sig_genes, collapse=', ')))

PAPER_GENES <- c('TSLP','BIRC5','S100B','MDK','S100P','RARRES3','BLNK','ACO1')
overlap <- intersect(sig_genes, PAPER_GENES)
cat(sprintf('Paper overlap: %d/%d — %s\n', length(overlap),
            length(PAPER_GENES), paste(overlap, collapse=', ')))

In [ ]:
p_vimp <- ggplot(head(vimp_df, 20),
                 aes(x=importance, y=reorder(gene, importance))) +
  geom_col(fill='#4DBBD5', alpha=0.8) +
  geom_vline(xintercept=0, colour='grey40') +
  labs(title='Survival SVM — Permutation Importance (Top 20)',
       x='Mean decrease in C-index', y=NULL) +
  theme_bw(base_size=10) +
  theme(plot.title=element_text(face='bold'))

p_vimp
ggsave(file.path(VIS_DIR, 'svm_permutation_importance.png'), p_vimp,
       dpi=150, width=8, height=6)
cat('Saved: svm_permutation_importance.png\n')

## Patient Stratification & Kaplan–Meier Curves

In [ ]:
rs_train <- predict(svm_final, newdata=train_df)$predicted
rs_t1    <- predict(svm_final, newdata=test1_df)$predicted
rs_t2    <- predict(svm_final, newdata=test2_df)$predicted
rs_t3    <- predict(svm_final, newdata=test3_df)$predicted

train_median_risk <- median(rs_train)
risk_label <- function(scores, cutoff) ifelse(scores >= cutoff, 'High Risk', 'Low Risk')

make_km_df <- function(scores, df) {
  data.frame(time=df$relapse_free_time, event=as.integer(df$relapse_free_event),
             risk_group=factor(risk_label(scores, train_median_risk),
                               levels=c('Low Risk','High Risk')))
}

km_plot <- function(df_km, title) {
  fit <- survfit(Surv(time, event) ~ risk_group, data=df_km)
  ggsurvplot(fit, data=df_km, pval=TRUE, conf.int=TRUE,
             palette=c('Low Risk'='#4DBBD5','High Risk'='#E64B35'),
             title=title, xlab='Time (days)', ylab='Relapse-Free Survival',
             legend.labs=c('Low Risk','High Risk'),
             ggtheme=theme_bw(base_size=9))$plot +
    theme(plot.title=element_text(face='bold', size=10))
}

km_plots <- list(
  km_plot(make_km_df(rs_train, train_df), 'Train'),
  km_plot(make_km_df(rs_t1,    test1_df), 'Test 1'),
  km_plot(make_km_df(rs_t2,    test2_df), 'Test 2'),
  km_plot(make_km_df(rs_t3,    test3_df), 'Test 3')
)
ggarrange(plotlist=km_plots, ncol=4, common.legend=TRUE, legend='bottom')

## Model Comparison Table

In [ ]:
# Build comparison — reads RSF and CoxBoost results if available
rsf_imp_path <- file.path(DATA_DIR, 'rsf_importance.csv')
gb_imp_path  <- file.path(DATA_DIR, 'gb_importance.csv')

comparison <- data.frame(
  Model  = c('Survival SVM'),
  Train  = round(ci_train, 4),
  Test_1 = round(ci_t1,    4),
  Test_2 = round(ci_t2,    4),
  Test_3 = round(ci_t3,    4),
  stringsAsFactors = FALSE
)
cat('C-index Comparison:\n')
print(comparison)

## Save Model & Results

In [ ]:
saveRDS(svm_final, file.path(MODEL_DIR, 'survivalsvm_model.rds'))
write.csv(vimp_df,    file.path(DATA_DIR, 'svm_importance.csv'),        row.names=FALSE)
write.csv(comparison, file.path(DATA_DIR, 'svm_model_comparison.csv'),  row.names=FALSE)

ggsave(file.path(VIS_DIR, 'svm_km_stratification.png'),
       ggarrange(plotlist=km_plots, ncol=4, common.legend=TRUE, legend='bottom'),
       dpi=150, width=20, height=5)

cat('=== Survival SVM Summary ===\n')
cat(sprintf('Best gamma.mu:   %.6f\n', best_gamma))
cat(sprintf('Best CV C-index: %.4f\n', best_cv_ci))
cat(sprintf('C-index — Train: %.4f | Test1: %.4f | Test2: %.4f | Test3: %.4f\n',
            ci_train, ci_t1, ci_t2, ci_t3))
cat('Saved: survivalsvm_model.rds, svm_importance.csv,\n')
cat('       svm_model_comparison.csv, svm_km_stratification.png\n')